In [ ]:
import pandas as pd
import yfinance as yf
import duckdb

In [ ]:
ar_adrs = [
    "YPF",    # YPF S.A. (NYSE)
    "GGAL",   # Grupo Financiero Galicia (NASDAQ)
    "BMA",    # Banco Macro (NYSE)
    "BBAR",   # BBVA Argentina (NYSE)
    "PAM",    # Pampa Energia (NYSE)
    "TEO",    # Telecom Argentina (NYSE)
    "CEPU",   # Central Puerto (NYSE)
    "LOMA",   # Loma Negra (NYSE)
    "CRESY",  # Cresud (NASDAQ)
    "IRS",    # IRSA Inversiones (NYSE)
    "SUPV",   # Grupo Supervielle (NYSE)
    # "DESP",   # Despegar.com (NYSE)
    "MELI",   # MercadoLibre (NASDAQ)
    "BIOX",   # Bioceres Crop Solutions (NASDAQ)
]


In [ ]:
# data = yf.download(ar_adrs, start="2018-01-01")
data = yf.download(ar_adrs, start="2018-01-01", group_by="ticker")

In [ ]:
data.info()

In [ ]:
data.columns.levels[1]

In [ ]:
duckdb.sql('SELECT * FROM bb')

In [ ]:
bb = data.T.dropna(how='all')

In [ ]:
bb = bb.T

In [ ]:
bb.info()

In [ ]:
def normalize_prices_long(df: pd.DataFrame) -> pd.DataFrame:
    """
    MultiIndex columns (ticker, field) -> long tidy table:
    date, ticker, open, high, low, close, adj_close, volume
    """
    out = (
        df.copy()
          .rename(columns={"Adj Close": "Adj_Close"}, level=1)
          .stack(level=0, future_stack=True)      # index: date, ticker
          .reset_index()
          .rename(columns={"level_0": "date", "level_1": "ticker"})
    )

    # standardize column names
    out.columns = [c.lower().replace(" ", "_") for c in out.columns]
    # if you renamed Adj Close -> Adj_Close above, this becomes adj_close
    return out

In [ ]:
cc = normalize_prices_long(bb)

In [ ]:
cc

In [ ]:
def build_ticker_dim(df: pd.DataFrame) -> pd.DataFrame:
    tickers = df.columns.get_level_values(0).unique()

    rows = []
    for t in tickers:
        sub = df[t]
        first_date = sub.dropna(how="all").index.min()
        last_date  = sub.dropna(how="all").index.max()
        has_data = pd.notna(first_date)

        rows.append({
            "ticker": t,
            "has_data": bool(has_data),
            "first_date": first_date,
            "last_date": last_date,
        })

    return pd.DataFrame(rows).sort_values(["has_data","ticker"], ascending=[False, True])

ticker_dim = build_ticker_dim(data)
print(ticker_dim)


In [ ]:
data.columns.levels[1]